In [12]:
import numpy as np 
import pandas as pd

df= pd.read_excel('india_states_urbanization.xlsx')
df_drinkingwater= pd.read_excel('Drinkingwater_Stats_Residence.xlsx')
df_sanitation= pd.read_excel('Sanitation_Stats_Residence.xlsx')

print(df.columns.tolist())

['States/UTs', 'Total (Figures in Thousands)', 'Urban', 'Rural', 'Urbanization_Rate', 'Urbanization_%', 'Area_sq_km', 'Population_actual', 'Population_Density']


In [33]:
df.columns = df.columns.str.strip()
df_sanitation.columns = df_sanitation.columns.str.strip()
df_sanitation.index = df_sanitation.index.astype(str).str.strip()

# Ensure consistent naming
df.rename(columns={
    "States/UTs": "state",
    "Urbanization_%": "urban_pct",
}, inplace=True)


df["urban_pct"] = df["urban_pct"] / 100



# Sanitation Risk-related categories
sanitation_urban_unimproved = df_sanitation.iat[0,3]
# print(sanitation_urban_unimproved) #verifying the correct value

sanitation_urban_no_facility = df_sanitation.iat[0,4]
# print(sanitation_urban_no_facility) 

sanitation_rural_unimproved = df_sanitation.iat[1,3]
sanitation_rural_no_facility = df_sanitation.iat[1,4]

# Water Risk-related categories
water_urban_unimproved = df_drinkingwater.iat[0,11]
water_urban_tanker = df_drinkingwater.iat[0, 8]
water_urban_rain = df_drinkingwater.iat[0, 7]

# print(water_urban_unimproved, water_urban_tanker, water_urban_rain ) #verifying values

water_rural_unimproved = df_drinkingwater.iat[1,11]
water_rural_tanker = df_drinkingwater.iat[1,8]
water_rural_rain =df_drinkingwater.iat[1,7]


sanitation_risk_urban = sanitation_urban_unimproved + sanitation_urban_no_facility
sanitation_risk_rural = sanitation_rural_unimproved + sanitation_rural_no_facility 

df["sanitation_risk"] = ( #the sanitation risk will be the same value throughout df
    df["urban_pct"] * sanitation_risk_urban +
    (1 - df["urban_pct"]) * sanitation_risk_rural
) / 100  # normalize


water_risk_urban = (
    water_urban_unimproved +
    water_urban_tanker +
    water_urban_rain
)

water_risk_rural = (
    water_rural_unimproved +
    water_rural_tanker +
    water_rural_rain
)

df["water_risk"] = ( #the water risk will be the same value throughout df
    df["urban_pct"] * water_risk_urban +
    (1 - df["urban_pct"]) * water_risk_rural
) / 100  # normalize


df["infra_risk"] = ( 
    df["sanitation_risk"] + df["water_risk"]
) / 2

df["density_infra"] = df["Population_Density"] * df["infra_risk"]  # Population density of each state interacting with the total infrastructure risk 


df["population_at_risk"] = df["Population_actual"] * df["infra_risk"] #infra_risk takes into account urban and rural components of population


final_df = df[[
    "state",
    "Population_actual",
    "Population_Density",
    "urban_pct",
    "sanitation_risk",
    "water_risk",
    "infra_risk",
    "density_infra",
    "population_at_risk"
]]


#saving final df to use wth rainfall data
final_df.to_excel("state_infrastructure_features.xlsx", index=False)

print(final_df) #verifying final df
print(final_df.columns.tolist())

                        state  Population_actual  Population_Density  \
0   Andaman & Nicobar Islands             400000           48.490726   
1              Andhra Pradesh           52787000          323.910215   
2           Arunachal Pradesh            1533000           18.306008   
3                       Assam           35043000          446.760499   
4                       Bihar          123083000         1307.127003   
5                  Chandigarh            1208000        10596.491230   
6                Chhattisgarh           29493000          218.156400   
7        Dadra & Nagar Haveli             608000         1238.289206   
8                 Daman & Diu             469000         4187.500000   
9                         Goa            1559000          421.123717   
10                    Gujarat           69788000          355.618516   
11                    Haryana           29483000          666.855152   
12           Himachal Pradesh            7394000          132.81